# Acquisition des données reprise de la correction du TD3, puis adaptée au TD4

In [1]:
# Recharge automatiquement nos fichiers .py (Document, Author, Corpus)
# quand on les modifie, sans redémarrer le noyau
%load_ext autoreload
%autoreload 2

In [28]:
import os
import json
import html
import urllib.request
import urllib.parse

import xmltodict
import pandas as pd
from datetime import datetime

from Document import Document, HackerNewsDocument, ArxivDocument
from Author import Author
from Corpus import Corpus
from DocumentFactory import DocumentFactory
from Nettoyage import supprimer_balises, nettoyer_texte

In [3]:
# la thématique choisie, à modifier librement
theme = "climate"
corpus = Corpus(theme)  
print(corpus)

Corpus : climate
Nombre de documents : 0
Nombre d'auteurs : 0


In [4]:

url_hn = "https://hn.algolia.com/api/v1/search?" + urllib.parse.urlencode(
    {"query": theme, "tags": "story", "hitsPerPage": 50})
print(url_hn)

with urllib.request.urlopen(url_hn) as reponse:
    d_hn = json.load(reponse)

https://hn.algolia.com/api/v1/search?query=climate&tags=story&hitsPerPage=50


In [5]:
print(d_hn.keys())           
print(d_hn["nbHits"])       
hits = d_hn["hits"]
print(len(hits))             
print(hits[0].keys())        

dict_keys(['exhaustive', 'exhaustiveNbHits', 'exhaustiveTypo', 'hits', 'hitsPerPage', 'nbHits', 'nbPages', 'page', 'params', 'processingTimeMS', 'processingTimingsMS', 'query', 'serverTimeMS'])
18567
50
dict_keys(['_highlightResult', '_tags', 'author', 'children', 'created_at', 'created_at_i', 'num_comments', 'objectID', 'points', 'story_id', 'title', 'updated_at', 'url'])


In [6]:

for hit in hits:
    texte = hit.get("story_text")
    if not texte:
        texte = hit.get("title", "")

    texte = html.unescape(texte).replace("<p>", " ").strip()
    texte = texte.replace("\n", " ")

    titre = hit.get("title", "")
    auteur = hit.get("author", "")
    date = datetime.strptime(hit["created_at"][:10], "%Y-%m-%d")
    url = hit.get("url")
    if not url:   # posts sans lien : on fabrique l'adresse du post
        url = "https://news.ycombinator.com/item?id=" + hit["objectID"]
    nb_commentaires = hit.get("num_comments", 0)
    
    # TD5 : la Factory crée le document Hacker News, puis on l'ajoute au corpus.
    doc = DocumentFactory.creer("Hacker News", titre, auteur, date, url, texte, nb_commentaires=nb_commentaires)
    
    # ajout au corpus
    corpus.add_document(doc)

    

print("Documents Hacker News :", corpus.ndoc)

Documents Hacker News : 50


In [7]:
url_arxiv = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode(
    {"search_query": "all:" + theme, "start": 0, "max_results": 50})
print(url_arxiv)

with urllib.request.urlopen(url_arxiv) as reponse:
    d_arxiv = xmltodict.parse(reponse.read())

http://export.arxiv.org/api/query?search_query=all%3Aclimate&start=0&max_results=50


In [8]:
print(d_arxiv.keys())               
print(d_arxiv["feed"].keys())        

entrees = d_arxiv["feed"].get("entry", [])
if isinstance(entrees, dict):
    entrees = [entrees]
    
print(len(entrees))

if entrees:                          
    print(entrees[0].keys())         

dict_keys(['feed'])
dict_keys(['@xmlns:opensearch', '@xmlns:arxiv', '@xmlns', 'id', 'title', 'updated', 'link', 'opensearch:itemsPerPage', 'opensearch:totalResults', 'opensearch:startIndex', 'entry'])
50
dict_keys(['id', 'title', 'updated', 'link', 'summary', 'category', 'published', 'arxiv:comment', 'arxiv:primary_category', 'author'])


In [9]:
for entree in entrees:
    texte = entree["summary"].replace("\n", " ").strip()

    # TD4 : les métadonnées
    titre = entree["title"].replace("\n", " ").strip()
    date = datetime.strptime(entree["published"][:10], "%Y-%m-%d")
    url = entree["id"]

    auteurs = entree["author"]
    if isinstance(auteurs, dict):
        auteurs = [auteurs]

    auteur = [a["name"] for a in auteurs]
    if auteur:
        auteur_principal = auteur[0]
        co_auteurs = auteur[1:]
    else:
        auteur_principal = "Auteur inconnu"
        co_auteurs = []

    # TD5 : la Factory crée le document arXiv, puis on l'ajoute au corpus.
    doc = DocumentFactory.creer("Arxiv", titre, auteur_principal, date, url, texte, co_auteurs=co_auteurs)
    corpus.add_document(doc)

print("Taille du corpus :", corpus.ndoc)

Taille du corpus : 100


In [10]:
     
print(corpus.id2doc[60])      
corpus.id2doc[60].afficher()  

Titre : Deep Learning for Climate Model Output Statistics |Auteur : Michael Steininger |Co_auteurs :Daniel Abel,Katrin Ziegler,Anna Krause,Heiko Paeth,Andreas Hotho
Titre : Deep Learning for Climate Model Output Statistics
Auteur : Michael Steininger
Date : 2020-12-09 00:00:00
URL : http://arxiv.org/abs/2012.10394v1
Texte : Climate models are an important tool for the assessment of prospective climate change effects but they suffer from systematic and representation errors, especially for precipitation. Model output statistics (MOS) reduce these errors by fitting the model output to observational data with machine learning. In this work, we explore the feasibility and potential of deep learning with convolutional neural networks (CNNs) for MOS. We propose the CNN architecture ConvMOS specifically designed for reducing errors in climate model outputs and apply it to the climate model REMO. Our results show a considerable reduction of errors and mostly improved performance compared to th

In [11]:
# Affichage des doc trié par date
corpus.afficher_par_date(10)

Titre : Climate simulations for 1880-2003 with GISS modelE
Auteur : J. Hansen
Date : 2006-10-16 00:00:00
URL : http://arxiv.org/abs/physics/0610109v2
Texte : We carry out climate simulations for 1880-2003 with GISS modelE driven by ten measured or estimated climate forcings. An ensemble of climate model runs is carried out for each forcing acting individually and for all forcing mechanisms acting together. We compare side-by-side simulated climate change for each forcing, all forcings, observations, unforced variability among model ensemble members, and, if available, observed variability. Discrepancies between observations and simulations with all forcings are due to model deficiencies, inaccurate or incomplete forcings, and imperfect observations. Although there are notable discrepancies between model and observations, the fidelity is sufficient to encourage use of the model for simulations of future climate change. By using a fixed well-documented model and accurately defining the 1

In [12]:
# Affichage des doc trié par titre
corpus.afficher_par_titre(10)

Titre : A critical step to reduce climate change
Auteur : mhandley
Date : 2019-05-15 00:00:00
URL : https://www.gatesnotes.com/Energy/A-critical-step-to-reduce-climate-change?WT.mc_id=20190515000825_Energy-Storage_BG-TW&WT.tsrc=BGTW&linkId=67393850
Texte : A critical step to reduce climate change
Source : Hacker News

Titre : A fractal climate response function can simulate global average temperature trends of the modern era and the past millennium
Auteur : J. H. van Hateren
Date : 2011-11-22 00:00:00
URL : http://arxiv.org/abs/1111.5177v3
Texte : A climate response function is introduced that consists of six exponential (low-pass) filters with weights depending as a power law on their e-folding times. The response of this two-parameter function to the combined forcings of solar irradiance, greenhouse gases, and SO2-related aerosols is fitted simultaneously to reconstructed temperatures of the past millennium, the response to solar cycles, the response to the 1991 Pinatubo volcanic eru

In [13]:
#for texte in df["texte"]:
  #  mots = texte.split(" ")
   # phrases = texte.split(".")
    #print(len(mots), "mots,", len(phrases), "phrases")

In [14]:

#print("Avant filtrage :", len(df))
#df = df[df["texte"].str.len() >= 20]
#print("Après filtrage :", len(df))

In [15]:
# TD3 3.4 adapté au TD4 : une chaîne avec tous les textes du corpus
chaine = " ".join(doc.texte for doc in corpus.id2doc.values())
print(len(chaine))
print(chaine[:500])

63594
Climate change: IPCC report is 'code red for humanity' Climate activist arrested after ProtonMail provided his IP address White House eyes subsidies for nuclear plants to help meet climate targets DOE announces breakthrough in residential cold climate heat pump technology Websites hosting major US climate reports taken down Project Vesta – Mitigating climate change with green sand beaches Former NOAA employees built Climate.us to preserve climate data and resources WA, NY and CA Governors Announ


In [16]:
print(list(corpus.authors.keys())[:10])

['perfunctory', 'kdunglas', 'pseudolus', 'josephscott', 'geox', 'QuickToBan', 'benwerd', 'rbanffy', 'antouank', 'scrum-treats']


In [17]:
# 2.4 : statistiques sur un auteur choisi par l'utilisateur
nom = input("Nom de l'auteur : ")

if nom not in corpus.authors:
    print("Auteur inconnu.")
else:
    #notre objet auteur
    auteur = corpus.authors[nom]                       

    print("Auteur :", auteur.name) # nom de l'auteur
    # Nombre de documents : déjà calculé par add()
    print("Nombre de documents :", auteur.ndoc)

    # Taille moyenne : on additionne la taille de chaque document
    total = 0
    for doc in auteur.production.values():     
        total += len(doc.texte)               
    moyenne = total / auteur.ndoc
    print("Taille moyenne :", round(moyenne, 1), "caractères")

Nom de l'auteur :  perfunctory


Auteur : perfunctory
Nombre de documents : 1
Taille moyenne : 54.0 caractères


In [18]:
# Sauvegarde et charger le corpus en csv
file = "corpus.csv"
corpus.save(file)

Corpus sauvegardé.


In [19]:
# Test du Singleton : deux appels à Corpus  doivent donner le même objet
c1 = Corpus("climate")
c2 = Corpus("autre nom")
print(c1 is c2)      
print(c2.nom)        

# Test de load : le corpus rechargé doit contenir 100 documents, pas 200
print(Corpus.load("corpus.csv"))

True
climate
Corpus : climate
Nombre de documents : 100
Nombre d'auteurs : 93


In [20]:
print(Corpus.load(file))

Corpus : climate
Nombre de documents : 100
Nombre d'auteurs : 93


In [21]:
# Vérifie que la Factory crée la bonne classe selon le type demandé.
d1 = DocumentFactory.creer("Hacker News", "Titre HN", "auteur1", None, "url1", "texte1", nb_commentaires=5)
d2 = DocumentFactory.creer("Arxiv", "Titre Arxiv", "auteur2", None, "url2", "texte2", co_auteurs=["A", "B"])
print(type(d1).__name__, "-", d1.getType())
print(type(d2).__name__, "-", d2.getType())

HackerNewsDocument - Hacker News
ArxivDocument - Arxiv


In [22]:
# Test de search (TD6 1.1)
passages = corpus.search("nuclear", 20)
print(len(passages), "passages trouvés")
for p in passages[:5]:
    print("...", p, "...")

4 passages trouvés
...  eyes subsidies for nuclear plants to help meet ...
...  Oil Giant of Fraud Nuclear energy is a vital p ...
... ars in Climate Tech Nuclear power is the best c ...
... t climate change Is nuclear power our best bet  ...


In [30]:
# Test de nettoyer_texte (TD6 2.2) : balises, majuscules, retour à la ligne, ponctuation, chiffres
texte = 'Voir <a href="https://fr.wikipedia.org/wiki/Climat_du_Rh%C3%B4ne" rel="nofollow">ce LIEN</a> et <i>ce texte</i> !\nEn 2023, le CO2 a augmenté de 3,5 %...'
print(nettoyer_texte(texte))

voir ce lien et ce texte en le co a augmenté de
